## 1. Importing Libraries

In [ ]:
import os 

file_path = "C:\\Users\\Tushar\\OneDrive\\Desktop\\DA_Projects\\Project_6\\Data_Cleaning_With_Python\\Dataset"
##os.getcwd()

os.chdir(file_path)

import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

# Force Pandas to display all floats with 2 decimal places
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 2. Reading & Inspecting Raw Data

In [ ]:
raw_data = pd.read_excel('Data_Cleaning_Project_Dataset.xlsx')
raw_data.head(2)

In [ ]:
raw_data.shape

In [ ]:
raw_data.info()

#### All cols will be inspected.

#### These Columns appeared objects require careful inspection 
1. `Age`
2. `Order_Date`
3. `MRP`

In [ ]:
raw_data.isnull().sum()

In [ ]:
raw_data.columns

In [ ]:
raw_data.index

## 3. Started Cleaning Raw_Data

In [ ]:
clean_data = raw_data.copy()
clean_data.head(2)

#### Checking cols with most null values

In [ ]:
null_count = clean_data.isnull().sum()
null_count[null_count>0].sort_values(ascending=False)

#### Inspecting 'Age', 'Order_Date', 'MRP' Cols

### `1. 'Age' Col`

In [ ]:
clean_data['Age'].info()

In [ ]:
len(clean_data[clean_data['Age'].isna()])

#### Checking data type inside 'Age' col

In [ ]:
clean_data['Age'].apply(type).value_counts()

In [ ]:
clean_data[clean_data['Age'].apply(type) == float]

In [ ]:
clean_data[clean_data['Age'].apply(type) == str]

In [ ]:
parsed_age = pd.to_numeric(clean_data['Age'], errors='coerce')

In [ ]:
parsed_age.describe()

In [ ]:
parsed_age.nsmallest(10)

In [ ]:
parsed_age.nlargest(10)

#### Problematic Age rows

In [ ]:
clean_data.loc[[10, 11, 15, 60, 65]]

#### Issues found in 'Age' col

1. **Mixed data types :** 62 `int`, 7 `float` and 3 `str` entries
2. **System Aggregate/Forecast Rows :** 4 bulk/summary rows with `NaN` ages (Index 57, 61, 67, 71)
3. **Structural problem identified :** `Female` appeared in the Age Col (Index 3, 7)
4. **Text Placeholder:** `'?'` used as missing value placeholder (Index 17)
5. **Invalid Ages, Duplicates & System Record:** Invalid ages `-5` & `145` (Index 10, 11, 15); `-5` is duplicated (Index 10 & 15; Demo/System record with `0` & `-1` age (Index 60 & 65) 

#### Solving structural problem with 'Age' col having value of 'Gender' col "Female"

In [ ]:
clean_data[clean_data["Age"] == "Female"]

#### Identify misaligned rows and shift their values one col right

In [ ]:
shifted_values = clean_data.loc[[3, 7], :].shift(1,axis=1).values
clean_data.loc[[3, 7], :] = shifted_values

In [ ]:
clean_data.iloc[[3, 7]]

#### Drop system/demo/duplicate rows

In [ ]:
rows_to_drop = [15, 57, 60, 61, 65, 67, 71]

clean_data = clean_data.drop(index=rows_to_drop)

#### Convert 'Age' col to numeric

In [ ]:
clean_data['Age'] = pd.to_numeric(clean_data['Age'], errors='coerce')

#### Converting invalid ages (-5, 145) to NaN

In [ ]:
clean_data.loc[clean_data['Age'].isin([-5, 145]), 'Age'] = pd.NA

#### Convert 'Age' col to 'Int64' dtype

In [ ]:
clean_data['Age'] = clean_data['Age'].astype('Int64')

In [ ]:
clean_data['Age'].info()

### `2. 'Order_Date' Col`

#### Checking data types in 'Order_Date' col

In [ ]:
clean_data['Order_Date'].apply(type).value_counts()

#### Checking 'str' rows in 'Order_Date' Col

In [ ]:
clean_data[clean_data['Order_Date'].apply(type) == str]['Order_Date']

#### Convert to dates safely; bad strings turn into NaT

In [ ]:
parsed_dates = pd.to_datetime(clean_data['Order_Date'], errors='coerce', utc=True)

#### Rows which failed conversion & converted to NaT

In [ ]:
clean_data.loc[clean_data['Order_Date'].notna() & parsed_dates.isna(), 'Order_Date']

#### Checking date's range

In [ ]:
valid_dates = parsed_dates.dropna()

print("Earliest date:", valid_dates.min())
print("Latest date:", valid_dates.max())

#### Finding valid dates

In [ ]:
parsed_dates.dt.year.value_counts()

In [ ]:
parsed_dates.count()

In [ ]:
clean_data[parsed_dates.dt.year != 2026]

#### Issues found in 'Order_Date' col & their solutions

1. **Invalid Calendar Date:** `'31-02-2026'` does not exist[12]; `Convert to NaT`
2. **Format Inconsistency:** Mixed timezone (`'2026-01-18T14:32:00Z'`) alongside standard datetimes[13]; `Convert to standard timezone-naive datetime`
3. **Legacy/Fallback System Date:** `2020-01-01` recorded as an impossible operational year[22]; `Real order with bad year, convert date to NaT`
4. **Demo/Test Record:** `2000-01-01` [55]; `whole row will be dropped`

#### Set non-2026 'Order_Date' to NaT

In [ ]:
clean_dates = parsed_dates.where(parsed_dates.dt.year == 2026, pd.NaT).dt.tz_localize(None)

#### Assign the finalized dates back to clean_data

In [ ]:
clean_data['Order_Date'] = clean_dates

#### Drop the demo/test row (Index 55)

In [ ]:
clean_data = clean_data.drop(index=55)

#### Checking 'Order_Date' after cleaning

In [ ]:
clean_data['Order_Date'].apply(type).value_counts()

In [ ]:
clean_data['Order_Date'].info()

### `3. 'MRP' Col`

In [ ]:
clean_data['MRP'].info()

In [ ]:
clean_data['MRP'].apply(type).value_counts()

In [ ]:
clean_data[clean_data['MRP'].apply(type) == str]

#### Replacing currency symbols like '$' and commas, then convert to float

In [ ]:
parsed_mrp = pd.to_numeric(clean_data['MRP'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')

In [ ]:
parsed_mrp.describe()

#### Issues found in 'MRP' col & their solutions

1. **Currency Formatting:** Index 11 contained string entry `'$999'` with currency formatting instead of numeric format.

In [ ]:
clean_data['MRP'] = parsed_mrp

In [ ]:
clean_data['MRP'].info()

In [ ]:
clean_data.info()

### `4. 'Discount' Col`

In [ ]:
clean_data['Discount'].info()

In [ ]:
clean_data['Discount'].apply(type).value_counts()

#### Check summary statistics and unique values for 'Discount' col

In [ ]:
clean_data['Discount'].describe()

In [ ]:
clean_data['Discount'].value_counts(dropna=False).sort_index()

#### Issues found in 'Discount' col & their solutions

1. **Extreme Outliers/Absolute Amount Entry:** Index 3 & 7 contained `3199.0`, which represented the discounted price for MRP `3999` rather than the fractional discount rate. Adjusted to the standard `0.20` (20%) discount rate.
2. **Keypunch Data Entry Typo:** Index 10 contained `1.2` (> 100% discount rate). Corrected to `0.20` (20%).

#### Inspect rows with invalid discount values (> 1.0)

In [ ]:
clean_data[clean_data['Discount'] > 1.0][['Order_ID', 'MRP', 'Discount']]

#### Correct keypunch typo (1.2 -> 0.20)

In [ ]:
clean_data.loc[clean_data['Discount'] == 1.2, 'Discount'] = 0.20

#### Correct absolute price entries (3199.0 -> 0.20)

In [ ]:
clean_data.loc[clean_data['Discount'] == 3199.0, 'Discount'] = 0.20

In [ ]:
clean_data['Discount'].describe()

In [ ]:
clean_data['Discount'].value_counts()

### `5. 'Tax' Col`

#### Check summary stats and unique value distribution for 'Tax' col

In [ ]:
clean_data['Tax'].describe()

In [ ]:
clean_data['Tax'].value_counts()

#### Inspect the row with 0.0 tax rate

In [ ]:
clean_data[clean_data['Tax'] == 0.0]

#### Issues found in 'Tax' col

1. **Zero Tax Entry (Staging/Test Row):** Index 70 contained `Tax = 0.00` associated with a `DELETE_ME_STAGING_TEST` record.

#### Drop the staging test row

In [ ]:
clean_data = clean_data.drop(index=70)

In [ ]:
clean_data['Tax'].describe()

In [ ]:
clean_data['Tax'].value_counts()

### `6. 'Quantity' Col`

In [ ]:
clean_data['Quantity'].info()

In [ ]:
clean_data['Quantity'].isna().sum()

In [ ]:
clean_data['Quantity'].apply(type).value_counts()

In [ ]:
clean_data['Quantity'].describe()

In [ ]:
clean_data['Quantity'].value_counts()

#### Issues found in 'Quantity' col & their solutions

1. **Missing Values (4 rows) :** Confirmed via same product sold entries that single-unit orders (`1`) represent the mode across all affected product categories. Imputed `NaN` with `1`.
2. **Negative Value (-1.0) :** Index 12 contained `-1.0` qty. Corrected using absolute value (`1`).
3. **Extreme Outlier/Keypunch Typo (1000.0) :** Index 14 contained `1000.0` qty. Typo corrected to `1`.
4. **Zero Quantity (0.0) :** Index 17 contained `0.0` for a delivered order. Corrected to minimum valid & mode order unit (`1`).

#### Impute missing NaN values with mode (1.0)

In [ ]:
clean_data['Quantity'] = clean_data['Quantity'].fillna(1.0)

#### Correct negative sign flip (-1.0 -> 1.0)

In [ ]:
clean_data.loc[clean_data['Quantity'] == -1.0, 'Quantity'] = 1.0

#### Correct keypunch outlier typo (1000.0 -> 1.0)

In [ ]:
clean_data.loc[clean_data['Quantity'] == 1000.0, 'Quantity'] = 1.0

#### Correct zero quantity placeholder (0.0 -> 1.0)

In [ ]:
clean_data.loc[clean_data['Quantity'] == 0.0, 'Quantity'] = 1.0

#### Convert data type to 'Int64'

In [ ]:
clean_data['Quantity'] = clean_data['Quantity'].astype('Int64')

In [ ]:
clean_data['Quantity'].info()

In [ ]:
clean_data['Quantity'].describe()

In [ ]:
clean_data['Quantity'].value_counts()

### `7. 'Shipping_Cost' Col`

In [ ]:
clean_data['Shipping_Cost'].info()

In [ ]:
clean_data['Shipping_Cost'].describe()

In [ ]:
clean_data['Shipping_Cost'].value_counts()

#### Issues found in 'Shipping_Cost' col & their solutions

1. **Extreme Outlier/Keypunch Typo (2500.0):** Index 16 contained `2500.0` for a single Classic T-Shirt order. Corrected to standard category rate (`50.0`).
2. **Zero Value Placeholder (0.0):** Index 17 contained `0.0` shipping fee for a delivered Classic T-Shirt. Imputed to standard category rate (`50.0`).

#### Correct keypunch typo (2500.0 -> 50.0)

In [ ]:
clean_data.loc[clean_data['Shipping_Cost'] == 2500.0, 'Shipping_Cost'] = 50.0

#### Impute zero shipping cost placeholder (0.0 -> 50.0)

In [ ]:
clean_data.loc[clean_data['Shipping_Cost'] == 0.0, 'Shipping_Cost'] = 50.0

In [ ]:
clean_data['Shipping_Cost'].describe()

In [ ]:
clean_data['Shipping_Cost'].value_counts()

### `8. 'Order_ID' Col`

In [ ]:
clean_data['Order_ID'].info()

In [ ]:
clean_data[clean_data['Order_ID'].isna()]

In [ ]:
clean_data['Order_ID'].nunique()

#### Issues found in 'Order_ID' col & their solutions

1. **Missing Order IDs (2 rows):** Index 3 & 7 had `None` values.
2. **Sequential Pattern Audit:** Surrounding records follow a strict chronological/index sequence (`O1001` to `O1010`).
3. **Action Taken:** Imputed Index 3 with `O1004` and Index 7 with `O1008`, restoring 100% uniqueness and sequence integrity.

#### Impute missing sequential Order_IDs based on exact index progression

In [ ]:
clean_data.loc[3, 'Order_ID'] = 'O1004'
clean_data.loc[7, 'Order_ID'] = 'O1008'

In [ ]:
len(clean_data[clean_data['Order_ID'].isna()])

In [ ]:
clean_data['Order_ID'].nunique()

#### Checking rows where 'Order_ID' does NOT match the expected 'Oxxxx' format

In [ ]:
invalid_order_ids = clean_data[~clean_data['Order_ID'].astype(str).str.match(r'^O\d{4}$')]
invalid_order_ids

#### Checking extra spaces

In [ ]:
spaces_order_ids = clean_data[clean_data['Order_ID'].astype(str).str.contains(r'^\s|\s$', regex=True)]
spaces_order_ids

### `9. 'Customer_ID' Col`

In [ ]:
clean_data['Customer_ID'].info()

In [ ]:
clean_data[clean_data['Customer_ID'].isna()]

#### Issues found in 'Customer_ID' col & their solutions

1. **Missing Customer IDs (3 rows) :** Indexes 0, 6 & 7 had `NaN` / missing values for `Customer_ID`.
2. **Sequential Pattern Audit :** Surrounding records follow a strict chronological/index sequence (`C001` through `C010`).
3. **Action Taken:** Imputed Index 0 with `C001`, Index 6 with `C007`, and Index 7 with `C008`, restoring sequence integrity and completeness.

#### Impute missing sequential Customer_IDs based on exact index progression

In [ ]:
clean_data.loc[0, 'Customer_ID'] = 'C001'
clean_data.loc[6, 'Customer_ID'] = 'C007'
clean_data.loc[7, 'Customer_ID'] = 'C008'

In [ ]:
len(clean_data[clean_data['Customer_ID'].isna()])

In [ ]:
clean_data['Customer_ID'].nunique()

#### Checking rows where 'Customer_ID' does NOT match the expected 'Cxxx' format

In [ ]:
invalid_rows = clean_data[~clean_data['Customer_ID'].astype(str).str.match(r'^C\d{3}$')]
invalid_rows

#### Checking extra spaces

In [ ]:
extra_spaces = clean_data[clean_data['Customer_ID'].astype(str).str.contains(r'^\s|\s$', regex=True)]
extra_spaces

### `10. 'Product_ID' Col`

In [ ]:
clean_data['Product_ID'].info()

In [ ]:
clean_data[clean_data['Product_ID'].isna()][['Order_ID', 'Product_ID', 'Product_Name', 'Category', 'Brand', 'SKU']]

In [ ]:
clean_data['Product_ID'].value_counts()

In [ ]:
clean_data['SKU'].value_counts()

In [ ]:
clean_data[clean_data['SKU'].isin(['AD-TS-001'])]['Product_ID'].unique()

#### Issues found in 'Product_ID' col & their solutions

1. **Missing Product IDs (3 rows):** Indexes 3 and 7 (`Winter Jacket`) were `NaN` (imputed to `P004`). Index 4 (`Classic T-Shirt`) was `NaN` (imputed to `P005`).
2. **Inconsistent Mapping & Outliers (`P001`, `P999`):** Cross-tabulation showed `SKU: AD-TS-001` had split entries (`P001`, `P005`, `P999`). Standardized all `AD-TS-001` entries to mode `P005`.

#### Impute missing Product_IDs based on Product_Name mapping

In [ ]:
clean_data.loc[(clean_data['Product_Name']=='Winter Jacket') & (clean_data['Product_ID'].isna()), 'Product_ID'] = 'P004'

#### Standardize all 'AD-TS-001' (P001, P999, NaN) to P005

In [ ]:
clean_data.loc[clean_data['SKU'] == 'AD-TS-001', 'Product_ID'] = 'P005'

In [ ]:
clean_data['Product_ID'].info()

### `11. 'Customer_Name' Col`

In [ ]:
clean_data['Customer_Name'].info()

In [ ]:
clean_data[clean_data['Customer_Name'].isna()]

#### Issues found in 'Customer_Name' col & their solutions

1. **Missing Values (2 rows):** Indexes 2 and 7 were missing (`NaN`); Imputed with `'Unknown'`.
2. **Formatting & Whitespace:** Potential leading/trailing spaces and non-standard capitalization; Applied `.str.strip()`, collapsed extra internal spaces, and enforced Title Case across all entries.
3. **Encoding Corruption (Mojibake):** UTF-8 accents (`é`, `ñ`, `ö`) rendered as Latin-1 artifacts (`RenÃ©`); Repaired via vectorized byte re-encoding (`.str.encode('latin-1').str.decode('utf-8')`).

#### Fill missing Customer_Names

In [ ]:
clean_data['Customer_Name'] = clean_data['Customer_Name'].fillna('Unknown')

#### Non-alphabetic characters (numbers, punctuation, symbols)

In [ ]:
bad_chars = clean_data[clean_data['Customer_Name'].fillna('').str.contains(r'[^a-zA-Z\s\-\'\u00C0-\u024F]', regex=True)]
bad_chars

#### Invisible/Extra Spacing

In [ ]:
bad_spaces = clean_data[clean_data['Customer_Name'].fillna('').str.contains(r'^\s|\s$|\s{2,}', regex=True)]
bad_spaces

#### Incorrect Casing (not Title Case)

In [ ]:
bad_casing = clean_data[clean_data['Customer_Name'].notna() & ~clean_data['Customer_Name'].str.istitle()]
bad_casing

#### Outlier Lengths (Names that are suspiciously short <= 2 chars or long >= 30 chars)

In [ ]:
bad_lengths = clean_data[clean_data['Customer_Name'].notna() & clean_data['Customer_Name'].str.len().isin([1, 2]) | (clean_data['Customer_Name'].str.len() > 30)]
bad_lengths

#### Repair UTF-8 Mojibake encoding corruption across Customer_Name

In [ ]:
clean_data['Customer_Name'] = (
    clean_data['Customer_Name']
    .astype(str)
    .str.encode("latin-1", errors="ignore")
    .str.decode("utf-8", errors="ignore")
)

#### Strip quotes/nicknames, normalize whitespace, and enforce Title Case

In [ ]:
clean_data['Customer_Name'] = (
    clean_data['Customer_Name']
    .str.replace(r'[\“\”"]', '', regex=True)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

#### Verification Check: Inspect previously corrupted rows (52, 53, 58, 59, 62, 64, 68)

In [ ]:
clean_data.loc[[52, 53, 58, 59, 62, 64, 68], ['Customer_ID', 'Customer_Name']]

### `12. 'Gender' Col`

In [ ]:
clean_data['Gender'].info()

In [ ]:
clean_data['Gender'].value_counts()

#### Issues found in 'Gender' col & their solutions

1. **Inconsistent Casing & Abbreviations:** Variations like `'MALE'`, `'M'`, and `'f'`; Standardized using string mapping (`'M'`/`'MALE'` $\rightarrow$ `'Male'`, `'f'` $\rightarrow$ `'Female'`) and Title Case formatting.

#### Standardize Gender values

In [ ]:
gender_map = {
    'MALE': 'Male',
    'M': 'Male',
    'm': 'Male',
    'FEMALE': 'Female',
    'F': 'Female',
    'f': 'Female'
}

#### Apply mapping and enforce Title Case

In [ ]:
clean_data['Gender'] = clean_data['Gender'].replace(gender_map).str.title()

### `13. 'Email' Col`

In [ ]:
clean_data['Email'].info()

In [ ]:
clean_data[clean_data['Email'].isna()]

In [ ]:
clean_data.head(2)

In [ ]:
bad_emails = clean_data[clean_data['Email'].notna() & ~clean_data['Email'].str.contains(r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$', regex=True, na=False)]
bad_emails

#### Issues found in 'Email' col & their solutions

1. **Sensitive/Privacy Issues :** Rows 53, 54, 56, 62, 63, 66, and 69 contained unmasked sensitive payload artifacts (e.g., `_CC:FAKE...`, `_DOB:FAKE...`) and unverified testing domains (`.test`); Marked `'Unknown'`.
2. **Truncated Domain Extensions :** Rows 10 (`amit.mishra@gmail`) and 11 (`vikram@corp`) lacked TLD extensions; Repaired deterministic domain syntax by appending `.com`.
3. **Missing & Placeholder Values :** Rows 2 & 7 (`'Unknown'`), row 9 (`'-'`), and row 6 (`NaN`); Standardized as `'Unknown'`.

#### Strip privacy leak payload suffixes (_CC:FAKE..., _DOB:FAKE..., etc.)

In [ ]:
clean_data['Email'] = clean_data['Email'].astype(str).str.replace(r'_.*$', '', regex=True)

#### Repair deterministic truncated domain extensions (.com)

In [ ]:
clean_data['Email'] = clean_data['Email'].replace({
    'amit.mishra@gmail': 'amit.mishra@gmail.com',
    'vikram@corp': 'vikram@corp.com'
})

#### Strip whitespace and lower-case email strings

In [ ]:
clean_data['Email'] = clean_data['Email'].str.strip().str.lower()

#### Standardize all unverified .test domains, missing values, and placeholders to 'Unknown'

In [ ]:
clean_data.loc[clean_data['Email'].str.endswith('.test', na=False), 'Email'] = 'Unknown'
clean_data['Email'] = clean_data['Email'].replace({'unknown': 'Unknown', '-': 'Unknown', 'nan': 'Unknown'})
clean_data['Email'] = clean_data['Email'].fillna('Unknown')

#### Inspect all previously flagged problematic rows

In [ ]:
flagged_rows = [2, 7, 9, 10, 11, 53, 54, 56, 62, 63, 66, 69]

print("\n")
print("--- Verified Email Column Fixes ---")
print("\n")

clean_data.loc[flagged_rows, ['Customer_ID', 'Customer_Name', 'Email']]

#### Verify no invalid entries remain in the entire Email column

In [ ]:
remaining_bad = clean_data[
    (clean_data['Email'] != 'Unknown') & 
    ~clean_data['Email'].str.contains(r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$', regex=True, na=False)
]
print(f"\nRemaining invalid email rows in dataset: {len(remaining_bad)}")

### `14. 'Phone' Col`

In [ ]:
clean_data['Phone'].info()

#### Strip whitespace

In [ ]:
phone_str = clean_data['Phone'].astype(str).str.strip()

#### Non-digit characters or wrong length (not 10 digits)

In [ ]:
invalid_length = clean_data[~phone_str.str.match(r'^\d{10}$')]
invalid_length

#### Invalid starting digit for Indian mobile network (Must start with 6, 7, 8, or 9)

In [ ]:
invalid_prefix = clean_data[phone_str.str.match(r'^[0-5]\d{9}$')]

#### Repeated single digit (e.g., '9999999999', '0000000000')

In [ ]:
repeated_digits = clean_data[phone_str.str.contains(r'^(\d)\1{9}$', regex=True)]

#### Sequential dummy patterns ('1234567890', '9876543210', '0123456789')

In [ ]:
dummy_sequences = ['1234567890', '9876543210', '0123456789', '0987654321']
sequential_dummies = clean_data[phone_str.isin(dummy_sequences)]

#### Combine all audit findings cleanly without duplication

In [ ]:
all_bad_phones = clean_data.loc[
    invalid_length.index.union(invalid_prefix.index)
                        .union(repeated_digits.index)
                        .union(sequential_dummies.index)
]

print(f"Total flagged phone issues across all rule layers: {len(all_bad_phones)}")
all_bad_phones[['Customer_ID', 'Customer_Name', 'Phone']]

#### Issues found in 'Phone' col & their solutions

1. **Sequential Dummy Patterns :** Row 0 contained a mock sequential pattern (`'9876543210'`); Standardized to `'Unknown'`.
2. **Missing Placeholders :** Rows 2 & 7 contained hyphen placeholders (`'-'`); Standardized to `'Unknown'`.
3. **Truncated Phone Numbers :** Row 10 contained an incomplete 9-digit string (`'987654321'`); Standardized to `'Unknown'`.

#### Overwrite invalid phone entries in clean_data directly

In [ ]:
clean_data.loc[all_bad_phones.index, 'Phone'] = 'Unknown'

In [ ]:
remaining_bad_phones = clean_data[
    (clean_data['Phone'] != 'Unknown') & (
        ~phone_str.str.match(r'^[6-9]\d{9}$') |
        phone_str.isin(dummy_sequences)
    )
]

print(f"Remaining invalid phone rows in dataset: {len(remaining_bad_phones)}")

### `15. 'City' Col`

In [ ]:
clean_data['City'].info()

In [ ]:
clean_data[clean_data['City'].isna()]

#### Issues found in 'City' col & their solutions

1. **Missing Value Imputation:** Rows with missing `City` values (`NaN`) were mapped using their corresponding `Pincode` and `State`. Index[1, 5]
2. **Casing Standardization:** Fixed lowercase/uppercase typos (e.g., `mumbai` $\rightarrow$ `Mumbai`, `BENGALURU` $\rightarrow$ `Bengaluru`).
3. **Geographic Synonym Normalization:** Standardized historical naming variants by mapping `Bengaluru` to **`Bengalore`**.

In [ ]:
clean_data[clean_data['Pincode'].isin([110001	, 400002])][['Order_ID', 'City', 'State', 'Pincode']]

In [ ]:
clean_data['City'].value_counts()

#### Impute missing City values based on Pincode / State context

In [ ]:
clean_data.loc[(clean_data['City'].isna()) & (clean_data['Pincode'] == 110001), 'City'] = 'Delhi'
clean_data.loc[(clean_data['City'].isna()) & (clean_data['Pincode'] == 400002), 'City'] = 'Mumbai'

#### Strip whitespace and apply Title Case to fix variations like 'mumbai'

In [ ]:
clean_data['City'] = clean_data['City'].astype(str).str.strip().str.title()

#### Standardize city naming variants (e.g., Bengaluru -> Bengalore)

In [ ]:
clean_data['City'] = clean_data['City'].replace({'Bengaluru': 'Bangalore'})

In [ ]:
clean_data['City'].value_counts()

In [ ]:
clean_data['City'].info()

### `16. 'State' Col`

In [ ]:
clean_data['State'].info()

In [ ]:
clean_data[clean_data['State'].isna()]

#### Issues found in 'State' col & their solutions

1. **Missing Value Imputation:** The missing `State` entry (`NaN`) associated with `City: Mumbai` and `Pincode: 400004` was deterministically imputed as **`Maharashtra`**.
2. **Abbreviation & Variant Standardization:** Converted state abbreviations and descriptive naming variants into canonical full names:
   - `MAHA` $\rightarrow$ `Maharashtra`
   - `KA` $\rightarrow$ `Karnataka`
   - `TN` $\rightarrow$ `Tamil Nadu`
   - `New Delhi` $\rightarrow$ `Delhi`

In [ ]:
clean_data['State'].value_counts()

#### Impute Missing State

In [ ]:
clean_data.loc[clean_data['State'].isna() & (clean_data['Pincode'] == 400004), 'State'] = 'Maharashtra'

#### Clean whitespace & apply Title Case

In [ ]:
clean_data['State'] = clean_data['State'].astype(str).str.strip().str.title()

#### Standardized Mapping

In [ ]:
state_mapping = {
    'Maha': 'Maharashtra',
    'Ka': 'Karnataka',
    'Tn': 'Tamil Nadu',
    'New Delhi': 'Delhi'
}

#### Apply map abbreviations

In [ ]:
clean_data['State'] = clean_data['State'].replace(state_mapping)

In [ ]:
clean_data['State'].value_counts()

### `17. 'Pincode' Col`

In [ ]:
clean_data['Pincode'].info()

In [ ]:
clean_data['Pincode'].value_counts()

#### Issues found in 'Pincode' col & their solutions

1. **Formatting & Normalization:** Standardized the column by stripping all whitespace and ensuring uniform 6-digit string representation (`object` dtype.
2. **Relational Cross-Validation (Cross-Tabulation Audit):** 
   - Generated a cross-tabulation matrix (`pd.crosstab`) between `City` and the 3-digit `Pincode` prefixes to programmatically verify regional integrity.

#### Convert to string, strip whitespace, remove internal spaces

In [ ]:
clean_data['Pincode'] = clean_data['Pincode'].astype(str).str.strip().str.replace(r'\s+', '', regex=True)

In [ ]:
clean_data['Pincode'].info()
clean_data['Pincode'].value_counts()

In [ ]:
cross_check = pd.crosstab(
    clean_data["City"],
    clean_data["Pincode"].astype(str).str[:3].rename("Pincode_Prefix"),
)

display(cross_check)

### `18. 'Product_Name' Col`

In [ ]:
clean_data['Product_Name'].info()

In [ ]:
clean_data['Product_Name'].value_counts()

#### Issues found in 'Product_Name' col & their solutions

1. **Encoding Artifact Correction:** Identified character encoding pollution (Mojibake) caused by byte-decoding mismatches (`Classic T-ShÃ©rt`, `Running Sneakersâ,,¢`) and corrected them to canonical product titles.

#### Correcting Encoding Artifacts

In [ ]:
product_encoding_fixes = {
    'Classic T-Shã®Rt': 'Classic T-Shirt',
    'Running Sneakersâ„¢': 'Running Sneakers'
}

#### Clean whitespace, title case, and apply encoding fixes

In [ ]:
clean_data['Product_Name'] = clean_data['Product_Name'].astype(str).str.strip().str.title()
clean_data['Product_Name'] = clean_data['Product_Name'].replace(product_encoding_fixes)

In [ ]:
clean_data['Product_Name'].value_counts()

### `19. 'Category' Col`

In [ ]:
clean_data['Category'].info()

In [ ]:
clean_data['Category'].value_counts()

#### Issues found in 'Category' col & their solutions

1. **Casing Standardization:** Identified a case-sensitivity discrepancy (`clothing` vs `Clothing`) and standardized all entries to proper title case to ensure accurate grouping.

#### Clean whitespace & title case

In [ ]:
clean_data['Category'] = clean_data['Category'].astype(str).str.strip().str.title()

In [ ]:
clean_data['Category'].value_counts()

### `20. 'Brand' Col`

In [ ]:
clean_data['Brand'].info()

In [ ]:
clean_data['Brand'].value_counts()

#### Issues found in 'Brand' col & their solutions

1. **Casing Anomaly Correction:** Identified a lowercase inconsistency (`adidas` vs `Adidas`) and standardized all records to maintain canonical brand naming conventions.

#### Clean whitespace & converting to capitalize case

In [ ]:
clean_data['Brand'] = clean_data['Brand'].astype(str).str.strip().str.capitalize()

In [ ]:
clean_data['Brand'].value_counts()

### `21. 'SKU' Col`

In [ ]:
clean_data['SKU'].info()

In [ ]:
clean_data['SKU'].value_counts()

#### Issues found in 'Brand' col & their solutions

#### Ensure string type and strip any potential whitespace

In [ ]:
clean_data['SKU'] = clean_data['SKU'].astype(str).str.strip()

### `22. 'Order_Status' Col`

In [ ]:
clean_data['Order_Status'].info()

In [ ]:
clean_data['Order_Status'].value_counts()

#### Issues found in 'Order_Status' col & their solutions

1. **Casing Correction:** Resolved lowercase anomaly (`shipped` vs `Shipped`) to ensure clean categorical grouping.

#### Clean whitespace & title case

In [ ]:
clean_data['Order_Status'] = clean_data['Order_Status'].astype(str).str.strip().str.title()

In [ ]:
clean_data['Order_Status'].value_counts()

### `23. 'Payment_Method' Col`

In [ ]:
clean_data['Payment_Method'].info()

In [ ]:
clean_data['Payment_Method'].value_counts()

#### Issues found in 'Payment_Method' col & their solutions

1. **Casing & Synonym Remediation:** Standardized lowercase case variations (`upi` $\rightarrow$ `UPI`) and cleaned up formatting to ensure accurate payment channel distribution tracking.

#### Clean whitespace & converting to Title case

In [ ]:
clean_data['Payment_Method'] = clean_data['Payment_Method'].astype(str).str.strip().str.title()

#### Fix specific case variations

In [ ]:
clean_data['Payment_Method'] = clean_data['Payment_Method'].replace({
    'Upi': 'UPI',
    'Googlepay': 'Google Pay'
})

In [ ]:
clean_data['Payment_Method'].value_counts()

## 4. Advance Data Cleaning

### `1. 'Customer Entity Consistencty Check'`

In [ ]:
customer_drift = clean_data.groupby('Customer_ID').agg({
    'Customer_Name': 'nunique',
    'Email': 'nunique',
    'Phone': 'nunique'
})

# Filter for any customer ID where any attribute has more than 1 unique value
inconsistent_customers = customer_drift[
    (customer_drift['Customer_Name'] > 1) | 
    (customer_drift['Email'] > 1) | 
    (customer_drift['Phone'] > 1)
]

print(f"Inconsistent Customer IDs found: {len(inconsistent_customers)}")
if len(inconsistent_customers) > 0:
    display(inconsistent_customers)
else:
    print("✔ All customer entities are 100% stable (clean 1-to-1 mapping).")

### `2. 'Product Hierarchy Consistency Check'`

In [ ]:
product_drift = clean_data.groupby('Product_ID').agg({
    'Product_Name': 'nunique',
    'Category': 'nunique',
    'Brand': 'nunique',
    'SKU': 'nunique'
})

# Filter for any product ID where any attribute has more than 1 unique value
inconsistent_products = product_drift[
    (product_drift['Product_Name'] > 1) | 
    (product_drift['Category'] > 1) | 
    (product_drift['Brand'] > 1) | 
    (product_drift['SKU'] > 1)
]

print(f"Inconsistent Product IDs found: {len(inconsistent_products)}")
if len(inconsistent_products) > 0:
    display(inconsistent_products)
else:
    print("✔ All product hierarchies are 100% consistent!")

### `3. 'Geographic Triad (City Vs State) Check'`

In [ ]:
city_state_map = clean_data.groupby('City')['State'].nunique()

# Find any city that maps to more than 1 state
ambiguous_cities = city_state_map[city_state_map > 1]

print(f"Cities mapping to multiple states: {len(ambiguous_cities)}")
if len(ambiguous_cities) > 0:
    display(clean_data[clean_data['City'].isin(ambiguous_cities.index)][['City', 'State']].drop_duplicates())
else:
    print("✔ All cities map uniquely and correctly to their respective states!")

### `3.1 'Geographic Triad (State Vs Pincode) Check'`

In [ ]:
# Extract pincode prefix (e.g., first 3 digits) or use full Pincode
pincode_state_map = clean_data.groupby('Pincode')['State'].nunique()

# Find any pincode mapping to more than 1 state
ambiguous_pincodes = pincode_state_map[pincode_state_map > 1]

print(f"Pincodes mapping to multiple states: {len(ambiguous_pincodes)}")
if len(ambiguous_pincodes) > 0:
    display(clean_data[clean_data['Pincode'].isin(ambiguous_pincodes.index)][['Pincode', 'State']].drop_duplicates())
else:
    print("✔ All Pincodes map uniquely and correctly to their respective states!")

### `4. 'Duplicate Integrity Audit'`

In [ ]:
# --- DUPLICATE INTEGRITY AUDIT ---
exact_dupes = clean_data.duplicated().sum()
duplicate_orders = clean_data['Order_ID'].duplicated().sum()

print(f"Exact identical rows found: {exact_dupes}")
print(f"Duplicate Order IDs found: {duplicate_orders}")

if exact_dupes > 0:
    display(clean_data[clean_data.duplicated(keep=False)])
else:
    print("✔ Zero duplicate rows. Dataset uniqueness is 100% verified!")

## 5. Data Transformation

### `1. 'Financial & Revenue Feature Engineering'`

In [ ]:
# 1. Absolute discount monetary value per unit
clean_data['Discount_Amount'] = (clean_data['MRP'] * clean_data['Discount']).round(2)

# 2. Final selling price per unit after discount
clean_data['Selling_Price'] = (clean_data['MRP'] - clean_data['Discount_Amount']).round(2)

# 3. Subtotal for the line item (Selling Price * Quantity)
clean_data['Item_Subtotal'] = (clean_data['Selling_Price'] * clean_data['Quantity']).round(2)

# 4. Actual tax monetary value applied on the subtotal
clean_data['Tax_Amount'] = (clean_data['Item_Subtotal'] * clean_data['Tax']).round(2)

# 5. Grand Total (Subtotal + Tax + Shipping Cost)
clean_data['Total_Invoice_Value'] = (clean_data['Item_Subtotal'] + clean_data['Tax_Amount'] + clean_data['Shipping_Cost']).round(2)

print("✔ Financial features engineered successfully!")
clean_data[['Order_ID', 'MRP', 'Discount', 'Tax', 'Quantity', 'Shipping_Cost', 
            'Discount_Amount', 'Selling_Price', 'Item_Subtotal', 'Tax_Amount', 'Total_Invoice_Value']].head(3)

### `2. 'Temporal Feature Engineering'`

In [ ]:
# Extract Month Name and Day Name
clean_data['Order_Month'] = clean_data['Order_Date'].dt.month_name()
clean_data['Order_Day'] = clean_data['Order_Date'].dt.day_name()

# Create a binary flag for weekend orders (1 for weekend, 0 for weekday)
clean_data['Is_Weekend'] = (clean_data['Order_Date'].dt.dayofweek >= 5).astype(int)

print("✔ Temporal features engineered successfully!")
clean_data[['Order_ID', 'Order_Date', 'Order_Month', 'Order_Day', 'Is_Weekend']].head(3)

### `3. 'Demographic Segmentation'`

In [ ]:
# Define bins and labels for age cohorts
bins = [0, 25, 40, 60, 100]
labels = ['Young Adult', 'Adult', 'Middle-Aged', 'Senior']

clean_data['Age_Group'] = pd.cut(clean_data['Age'], bins=bins, labels=labels, right=True)

print("✔ Demographic segmentation engineered successfully!")
clean_data[['Customer_ID', 'Age', 'Age_Group']].head(3)

### `4. 'Reordering Columns for Logical Flow'`

In [ ]:
desired_column_order = [
    # 1. Order Details
    'Order_ID', 'Order_Date',
    
    # 2. Customer Profile (Age Group right beside Age!)
    'Customer_ID', 'Customer_Name', 'Age', 'Age_Group', 'Gender', 'Email', 'Phone',
    
    # 3. Geography
    'City', 'State', 'Pincode',
    
    # 4. Product Hierarchy
    'Product_ID', 'Product_Name', 'Category', 'Brand', 'SKU',
    
    # 5. Financials & Quantities
    'MRP', 'Discount', 'Discount_Amount', 'Selling_Price', 
    'Quantity', 'Item_Subtotal', 'Tax', 'Tax_Amount', 
    'Shipping_Cost', 'Total_Invoice_Value',

    # 6. Fulfillment & Payment
    'Payment_Method', 'Order_Status',
    
    # 7. Temporal Metadata
    'Order_Month', 'Order_Day', 'Is_Weekend'
]

# Reindex the dataframe to match our professional layout
clean_data = clean_data[desired_column_order]

print("✔ Columns successfully reordered into professional thematic groups!")
print(f"Total columns managed: {len(clean_data.columns)}")

### `5. 'Resetting Index'`

In [ ]:
clean_data = clean_data.reset_index(drop=True)

### `6. 'Auditing New Columns'`

In [ ]:
print("\n--- Newly Engineered Columns Audit ---")
new_cols = [
    'Age_Group', 'Discount_Amount', 'Selling_Price', 'Item_Subtotal', 
    'Tax_Amount', 'Total_Invoice_Value', 'Order_Month', 
    'Order_Day', 'Is_Weekend'
]
display(clean_data[new_cols].info())

In [ ]:
clean_data[new_cols].head(3)

#### Unifying Financial Data Types

In [ ]:
financial_cols_to_standardize = ['Item_Subtotal', 'Tax_Amount', 'Total_Invoice_Value']

for col in financial_cols_to_standardize:
    clean_data[col] = clean_data[col].astype('float64')

print("✔ All financial metrics successfully standardized to standard float64!")

In [ ]:
clean_data[new_cols].info()

## 6. Data Quality Showcase: 17-Point Cleaning Framework

This project demonstrates end-to-end mastery of data hygiene by identifying, auditing, and programmatically resolving all **17 industry-standard categories of data quality problems** within a single production-grade dataset.

| # | Data Quality Category | Status | Concrete Solution Implemented in Pipeline |
|---|---|---|---|
| **1** | **Missing / Incomplete Data** | ✅ Resolved | Audited null distributions; programmatically imputed missing keys (`Order_ID`, `Customer_ID`) and category modes (`Quantity`). |
| **2** | **Duplicate Data** | ✅ Resolved | Executed duplicate row checks and unique ID assertions across primary keys (`Order_ID`, `Customer_ID`). |
| **3** | **Inconsistent Formatting** | ✅ Resolved | Standardized text fields using `.str.strip()`, string collapse regex, and canonical capitalization rules. |
| **4** | **Inconsistent Categorical Values** | ✅ Resolved | Mapped and consolidated messy variants (`MALE`, `M` $\rightarrow$ `Male`; `upi` $\rightarrow$ `UPI`). |
| **5** | **Incorrect Data Types** | ✅ Resolved | Harmonized datatypes (cast `Age` and `Quantity` to nullable extension type `Int64`, unified financial metrics to `float64`). |
| **6** | **Date & Time Problems** | ✅ Resolved | Parsed mixed datetimes, eliminated out-of-bounds historical dates (`2020`), corrected impossible calendar dates (`31-02-2026`), and stripped timezones. |
| **7** | **Numerical Data Problems** | ✅ Resolved | Corrected negative sign flips (`Quantity = -1`), impossible zero-value placeholders, and extreme keypunch outliers (`Shipping_Cost = 2500`). |
| **8** | **Text / String Problems** | ✅ Resolved | Cleaned typos, invisible spacing, and punctuation artifacts across textual metadata columns. |
| **9** | **Structural Problems** | ✅ Resolved | Fixed shifted row misalignments (where category attributes leaked into adjacent columns) and reordered 32 columns into logical thematic blocks. |
| **10** | **Relationship / Integrity Problems** | ✅ Resolved | Verified relational integrity via cross-tabulation audits across the geographic triad (`City`, `State`, `Pincode`) and product hierarchies (`Product_ID`, `SKU`). |
| **11** | **Logical / Business Rule Errors** | ✅ Resolved | Engineered dependent financial aggregates (`Discount_Amount`, `Selling_Price`, `Item_Subtotal`, `Tax_Amount`, `Total_Invoice_Value`) following strict retail logic. |
| **12** | **Outliers & Anomalies** | ✅ Resolved | Identified and sanitized extreme pricing/discount overrides (e.g., fractional discount rate errors like `1.2` and absolute price entries stored as rates). |
| **13** | **Unit / Measurement Problems** | ✅ Resolved | Harmonized numerical currency and percentage representations to uniform unit standards. |
| **14** | **Encoding / Character Problems** | ✅ Resolved | Repaired UTF-8 Mojibake corruption artifacts (`RenÃ©`, `Classic T-ShÃ©rt`) using vectorised Latin-1 byte re-encoding. |
| **15** | **Data Granularity Problems** | ✅ Resolved | Ensured strict transaction-level granularity by isolating and dropping aggregate/forecast summary rows. |
| **16** | **Irrelevant / Redundant Data** | ✅ Resolved | Filtered and purged system staging records, demo templates, and test check rows (`DELETE_ME_STAGING_TEST`). |
| **17** | **Sensitive / Privacy Issues** | ✅ Resolved | Masked and neutralized unverified testing payloads, exposed testing domains (`.test`), and sanitized privacy leakage suffixes (`_CC:FAKE...`). |

## 7. Data Export

In [ ]:
filename = "Dataset_cleaned_with_pandas.csv"

#### Export the fully cleaned, transformed, and reordered dataframe

In [ ]:
#clean_data.to_csv(filename, index=False)

#print(f"✔ Successfully exported the Dataset_cleaned_with_pandas with {len(clean_data)} rows and {len(clean_data.columns)} columns to '{filename}'!")